## Sqlite3와 Pydantic

- SQLite3에서 가져온 데이터는 보통 tuple이나 dict 형태인데, 이를 Pydantic 모델로 변환하면 코드의 가독성과 안정성이 비약적으로 상승

In [ ]:
import sqlite3
from pydantic import BaseModel, ConfigDict
from typing import Optional

# 1. Pydantic 모델 정의 (데이터 스키마)
class User(BaseModel):
    id: int
    name: str
    email: str
    age: Optional[int] = None

    # SQLite의 Row 객체를 dict처럼 다루기 위한 설정 (Pydantic v2 기준)
    model_config = ConfigDict(from_attributes=True)

# 2. 데이터베이스 초기화 및 샘플 데이터 삽입 (강의용 세팅)
def init_db():
    conn = sqlite3.connect(":memory:")  # 테스트를 위해 메모리 DB 사용
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            name TEXT,
            email TEXT,
            age INTEGER
        )
    """)
    cursor.execute("INSERT INTO users VALUES (1, 'Alice', 'alice@example.com', 25)")
    cursor.execute("INSERT INTO users VALUES (2, 'Bob', 'bob@example.com', NULL)")
    conn.commit()
    return conn

# 3. 데이터 조회 및 Pydantic 변환 함수
def get_users(conn):
    # 결과를 dict 형태로 가져오기 위해 row_factory 설정
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users")

    rows = cursor.fetchall()

    # DB 행(Row) 데이터를 Pydantic 객체 리스트로 변환
    return [User.model_validate(dict(row)) for row in rows]

# --- 실행부 ---
connection = init_db()
user_list = get_users(connection)

for user in user_list:
    print(f"ID: {user.id} | 이름: {user.name} | 이메일: {user.email}")
    # Pydantic을 쓰면 user['name'] 대신 user.name (도트 연산자) 사용 가능!

ID: 1 | 이름: Alice | 이메일: alice@example.com
ID: 2 | 이름: Bob | 이메일: bob@example.com


## 실습과제

- https://github.com/ancestor9/data/blob/main/customers.csv 을 읽어와서 pydantic 에 저장하라
-- gradio로 외부 URL 로 웹서비스해라 (CRUD)

In [ ]:
import pandas as pd
from pydantic import BaseModel, Field
from typing import List, Optional

# Define the Pydantic model for customer data
class Customer(BaseModel):
    customer_id: str = Field(alias='고객ID')
    gender: str = Field(alias='성별')
    payment_method: str = Field(alias='결제수단')
    residence: str = Field(alias='거주지')
    membership_level: str = Field(alias='회원등급')
    satisfaction: int = Field(alias='만족도') # Corrected type from str to int
    recent_access_time_hour: int = Field(alias='최근접속시간(시)')
    preferred_product_temperature: float = Field(alias='선호제품군_적정온도')
    age: int = Field(alias='나이')
    purchase_quantity: int = Field(alias='구매수량')
    total_purchase_amount: int = Field(alias='총결제금액')

# URL of the CSV file
csv_url = "https://raw.githubusercontent.com/ancestor9/data/main/customers.csv"

try:
    # Read the CSV into a pandas DataFrame
    df = pd.read_csv(csv_url)
    print(f"CSV file loaded successfully. Number of records: {len(df)}")
    print("\nDataFrame columns:", df.columns.tolist()) # Added to inspect columns

    # Convert DataFrame records to Pydantic model instances
    # Using .to_dict('records') to get a list of dictionaries, then validating each with Pydantic
    customer_pydantic_list: List[Customer] = [Customer.model_validate(record) for record in df.to_dict('records')]

    print("\nFirst 5 customer Pydantic objects:")
    for i, customer in enumerate(customer_pydantic_list[:5]):
        print(customer.model_dump_json(indent=2))

except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure the CSV URL is correct and the Pydantic model matches the CSV columns.")


CSV file loaded successfully. Number of records: 10000

DataFrame columns: ['고객ID', '성별', '결제수단', '거주지', '회원등급', '만족도', '최근접속시간(시)', '선호제품군_적정온도', '나이', '구매수량', '총결제금액']

First 5 customer Pydantic objects:
{
  "customer_id": "CUST_0001",
  "gender": "남성",
  "payment_method": "휴대폰결제",
  "residence": "대구",
  "membership_level": "Gold",
  "satisfaction": 2,
  "recent_access_time_hour": 1,
  "preferred_product_temperature": 27.0,
  "age": 34,
  "purchase_quantity": 7,
  "total_purchase_amount": 760355
}
{
  "customer_id": "CUST_0002",
  "gender": "여성",
  "payment_method": "계좌이체",
  "residence": "대구",
  "membership_level": "Gold",
  "satisfaction": 5,
  "recent_access_time_hour": 15,
  "preferred_product_temperature": 24.6,
  "age": 20,
  "purchase_quantity": 42,
  "total_purchase_amount": 727001
}
{
  "customer_id": "CUST_0003",
  "gender": "여성",
  "payment_method": "신용카드",
  "residence": "광주",
  "membership_level": "Gold",
  "satisfaction": 1,
  "recent_access_time_hour": 6,
  "preferred_